# REVE на турецком наборе ЭЭГ: эпилепсия против нормы

| Поле | Значение |
|------|----------|
| **Автор** | Захар Санников (zs) |
| **Дата** | 2026-10-18 |
| **Гипотеза** | Г1 (ЭЭГ) |
| **Цель** | Проверить, что базовая модель REVE запускается на настоящих записях с эпилепсией, и сравнить её признаки с простыми (мощности ритмов) |
| **Данные** | Turkish Epilepsy EEG Dataset (Kaggle `buraktaci/turkish-epilepsy`, Tasci et al., Information Fusion, 2023): 50 эпилепсия + 71 норма (здоровые), 500 Гц, CC BY-NC-ND 4.0 |
| **Модель** | REVE (`brain-bzh/reve-base`, 69 млн параметров), веса **заморожены**, поверх — логистическая регрессия |
| **Деление обучение/тест** | **По пациентам**: одна запись = один человек = одна строка признаков. 5 частей × 10 повторов (RepeatedStratifiedKFold, seed 0) |
| **Среда** | Kaggle, GPU Tesla T4 |
| **Журнал** | строка в `experiments_log.md`: добавить |

> Блокнот выполнялся на Kaggle; выводы ячеек — фактические выводы того запуска (печать прогресса в ячейке признаков сокращена до каждого 10-го файла).

## 1. Данные

- Файлы `.mat`: в `epilepsy/` 50 файлов, в `normal/` 71. Один файл = один человек.
- Внутри: cell-массив `data` формы (куски × 36 каналов), каждая ячейка — 7 500 отсчётов = **15 с при 500 Гц**. `label` у всех кусков файла одинаковая.
- Длина записей: медиана 72 куска (~18 мин) у эпилепсии и 75 (~19 мин) у нормы.
- Каналы: 1–19 — относительно ушного электрода (FP1A1, FP2A2, F3A1 …), 20–35 — биполярные пары, 36-й авторы не использовали. **Берём только первые 19**: REVE нужны координаты электродов, а у биполярной пары одной точки нет.

In [ ]:
import os, glob, torch
from transformers import AutoModel

pos_bank = AutoModel.from_pretrained("brain-bzh/reve-positions", trust_remote_code=True)
model = AutoModel.from_pretrained("brain-bzh/reve-base", trust_remote_code=True).eval().cuda()
print("GPU:", torch.cuda.get_device_name(0))

GPU: Tesla T4

In [ ]:
import scipy.io
f = sorted(glob.glob("/kaggle/input/**/epilepsy/*.mat", recursive=True))[0]
print(f)
m = scipy.io.loadmat(f)
print({k: getattr(v, "shape", None) for k, v in m.items()})
d, lab = m["data"], m["label"]
print("dtype:", d.dtype)
print("ячейка [0,0]:", type(d[0, 0]), d[0, 0].shape, d[0, 0].dtype)
print("метки:", lab.ravel()[:10], "уникальные:", set(lab.ravel().tolist()))
print("мин/макс:", float(d[0, 0].min()), float(d[0, 0].max()))

/kaggle/input/datasets/buraktaci/turkish-epilepsy/Turkish Epilepsy Dataset/epilepsy/1_epilepsy (1).mat
{'__header__': None, '__version__': None, '__globals__': None, 'data': (48, 36), 'label': (1, 48)}
dtype: object
ячейка [0,0]: <class 'numpy.ndarray'> (7500, 1) float64
метки: [1 1 1 1 1 1 1 1 1 1] уникальные: {1}
мин/макс: -4038.366126344701 6118.232864179447

## 2. Предобработка и признаки

Для каждого 15-секундного куска:
1. Полосовой фильтр 0,5–45 Гц (Баттерворт 4-го порядка, `sosfiltfilt`).
2. **Простые признаки:** мощности ритмов по Уэлчу — дельта 1–4, тета 4–8, альфа 8–13, бета 13–30, гамма 30–45 Гц; абсолютные (логарифм) + относительные. 19 × 5 × 2 = 190 чисел.
3. **REVE:** переход на 200 Гц, нормировка каждого канала в окне (среднее 0, разброс 1) — **абсолютную амплитуду модель не видит**. Выход REVE: (окна, 19 каналов, 16 отрезков времени, 512). Два варианта сводки:
   - «по каналам» — среднее по времени, каналы сохраняются: 19 × 512 = 9 728 чисел;
   - «общее среднее» — среднее по всему: 512 чисел.
4. Признаки кусков усредняются → **один вектор на человека**.

Для проверки на перекосы сохраняются число кусков и медианный уровень сигнала (амплитуда, мкВ).

In [ ]:
import numpy as np, scipy.signal as ss, time

ROOT = "/kaggle/input/datasets/buraktaci/turkish-epilepsy/Turkish Epilepsy Dataset"
files = [(f, 1) for f in sorted(glob.glob(f"{ROOT}/epilepsy/*.mat"))] + \
        [(f, 0) for f in sorted(glob.glob(f"{ROOT}/normal/*.mat"))]
print("файлов:", len(files))

# первые 19 каналов (относительно уха); T3,T4,T5,T6 = T7,T8,P7,P8 в новой системе
NAMES = ["Fp1","Fp2","F3","F4","Fz","C3","C4","Cz","P3","P4","Pz","O1","O2","F7","F8","T7","T8","P7","P8"]
try:
    POS = pos_bank(NAMES)
except Exception as e:
    print("новые названия не подошли:", e)
    NAMES = ["Fp1","Fp2","F3","F4","Fz","C3","C4","Cz","P3","P4","Pz","O1","O2","F7","F8","T3","T4","T5","T6"]
    POS = pos_bank(NAMES)
POS = POS.cuda(); C = len(NAMES)
print("позиции:", tuple(POS.shape))

SOS = ss.butter(4, [0.5, 45], btype="band", fs=500, output="sos")
BANDS = [(1,4), (4,8), (8,13), (13,30), (30,45)]

def pool(out):
    if isinstance(out, (tuple, list)): out = out[0]
    if hasattr(out, "last_hidden_state"): out = out.last_hidden_state
    B, D = out.shape[0], out.shape[-1]
    if out.dim() == 4:                              # (B, каналы, время, D)
        per_ch = out.mean(2)
    elif out.dim() == 3 and out.shape[1] % C == 0:  # (B, каналы*время, D), каналы идут первыми
        per_ch = out.reshape(B, C, -1, D).mean(2)
    else:
        raise ValueError(f"неожиданная форма выхода {tuple(out.shape)}")
    return per_ch.reshape(B, -1), out.reshape(B, -1, D).mean(1)

F_ch, F_gl, F_bp, Y, N_SEG, AMP = [], [], [], [], [], []
t0 = time.time()
for i, (f, y) in enumerate(files):
    d = scipy.io.loadmat(f)["data"]
    X = np.stack([np.stack([d[s, c].ravel() for c in range(C)]) for s in range(d.shape[0])])  # (куски, 19, 7500)
    X = ss.sosfiltfilt(SOS, X, axis=-1)
    AMP.append(np.median(X.std(-1)))

    # простые признаки: мощности ритмов (абсолютные в логарифме + относительные)
    fr, P = ss.welch(X, fs=500, nperseg=1000, axis=-1)
    bp = np.stack([P[..., (fr >= a) & (fr < b)].mean(-1) for a, b in BANDS], -1)  # (куски, 19, 5)
    rel = bp / bp.sum(-1, keepdims=True)
    F_bp.append(np.concatenate([np.log(bp).mean(0).ravel(), rel.mean(0).ravel()]))

    # REVE: 200 Гц, нормировка каждого окна по каналу
    Xr = ss.resample_poly(X, 2, 5, axis=-1)                       # 7500 -> 3000 отсчётов
    Xr = (Xr - Xr.mean(-1, keepdims=True)) / (Xr.std(-1, keepdims=True) + 1e-8)
    ch, gl = [], []
    with torch.no_grad():
        for b in range(0, len(Xr), 16):
            x = torch.tensor(Xr[b:b+16], dtype=torch.float32).cuda()
            out = model(x, POS.expand(x.size(0), -1, -1))
            if i == 0 and b == 0:
                o = out[0] if isinstance(out, (tuple, list)) else getattr(out, "last_hidden_state", out)
                print("форма выхода REVE:", tuple(o.shape))
            a1, a2 = pool(out); ch.append(a1.cpu()); gl.append(a2.cpu())
    F_ch.append(torch.cat(ch).mean(0).numpy()); F_gl.append(torch.cat(gl).mean(0).numpy())
    Y.append(y); N_SEG.append(len(X))
    if (i + 1) % 10 == 0 or i + 1 == len(files):
        print(f"{i+1}/{len(files)} {time.time()-t0:.0f} с")

np.savez("/kaggle/working/feats.npz", ch=np.array(F_ch), gl=np.array(F_gl), bp=np.array(F_bp),
         y=np.array(Y), nseg=np.array(N_SEG), amp=np.array(AMP))
print("готово")

файлов: 121
позиции: (19, 3)
форма выхода REVE: (16, 19, 16, 512)
10/121 47 с
20/121 100 с
30/121 135 с
40/121 202 с
50/121 252 с
60/121 310 с
70/121 358 с
80/121 403 с
90/121 444 с
100/121 485 с
110/121 525 с
120/121 565 с
121/121 569 с
готово

## 3. Модель и метрики

Логистическая регрессия (StandardScaler → LogisticRegression) на векторах людей. Оценка — ROC AUC, среднее ± разброс по 50 разбиениям (5 частей × 10 повторов), разбиение по людям.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score

z = np.load("/kaggle/working/feats.npz"); y = z["y"]
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=0)

print("Проверка на перекосы между группами:")
for name in ["nseg", "amp"]:
    print(f"  {name}: эпилепсия {np.median(z[name][y==1]):.1f}, норма {np.median(z[name][y==0]):.1f}")

for feat, title in [("bp", "Мощности ритмов"), ("gl", "REVE, общее среднее"), ("ch", "REVE, по каналам")]:
    for Cr in [0.001, 0.01, 0.1, 1]:
        clf = make_pipeline(StandardScaler(), LogisticRegression(C=Cr, max_iter=5000))
        s = cross_val_score(clf, z[feat], y, cv=cv, scoring="roc_auc")
        print(f"{title:22s} C={Cr:<6} ROC AUC = {s.mean():.3f} ± {s.std():.3f}")

Проверка на перекосы между группами:
  nseg: эпилепсия 72.0, норма 75.0
  amp: эпилепсия 13.0, норма 8.9
Мощности ритмов        C=0.001  ROC AUC = 0.867 ± 0.072
Мощности ритмов        C=0.01   ROC AUC = 0.890 ± 0.062
Мощности ритмов        C=0.1    ROC AUC = 0.884 ± 0.059
Мощности ритмов        C=1      ROC AUC = 0.890 ± 0.057
REVE, общее среднее    C=0.001  ROC AUC = 0.882 ± 0.068
REVE, общее среднее    C=0.01   ROC AUC = 0.898 ± 0.059
REVE, общее среднее    C=0.1    ROC AUC = 0.910 ± 0.056
REVE, общее среднее    C=1      ROC AUC = 0.914 ± 0.055
REVE, по каналам       C=0.001  ROC AUC = 0.940 ± 0.040
REVE, по каналам       C=0.01   ROC AUC = 0.939 ± 0.041
REVE, по каналам       C=0.1    ROC AUC = 0.936 ± 0.043
REVE, по каналам       C=1      ROC AUC = 0.932 ± 0.044

### Проверка на перекосы

Уровень сигнала у эпилепсии в 1,5 раза выше (13,0 против 8,9 мкВ). Проверяем, насколько хорошо группы различаются **только по амплитуде** и по форме спектра без амплитуды.

In [ ]:
from sklearn.metrics import roc_auc_score
print("AUC только по амплитуде:", round(roc_auc_score(y, z["amp"]), 3))
rel = z["bp"][:, 95:]   # только относительные мощности (без амплитуды)
for Cr in [0.01, 0.1, 1]:
    s = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(C=Cr, max_iter=5000)), rel, y, cv=cv, scoring="roc_auc")
    print(f"Относительные мощности C={Cr}: {s.mean():.3f} ± {s.std():.3f}")

AUC только по амплитуде: 0.787
Относительные мощности C=0.01: 0.823 ± 0.078
Относительные мощности C=0.1: 0.806 ± 0.079
Относительные мощности C=1: 0.787 ± 0.085

## 4. Выводы

| Признаки | ROC AUC (лучшее C) |
|---|---|
| Только амплитуда | 0,79 |
| Относительные мощности ритмов (без амплитуды) | 0,82 |
| Мощности ритмов (все) | 0,89 ± 0,06 |
| REVE, общее среднее | 0,91 ± 0,06 |
| **REVE, по каналам** | **0,94 ± 0,04** |

1. **REVE работает на клинической ЭЭГ с эпилепсией**: весь путь от файла `.mat` до метрики на уровне человека собран, 121 запись считается ~10 минут на T4.
2. **REVE лучше простых признаков** (0,94 против 0,89). Разница сопоставима с разбросом, преимущество есть, но пока не очень надёжное.
3. **Каналы нужно сохранять**: среднее по каналам теряет информацию о том, где в мозге изменения (0,91 против 0,94).
4. **В наборе есть перекос**: одна амплитуда даёт 0,79. Группы различаются чем-то кроме эпилепсии (возраст, аппаратура, условия записи) — или это свойство болезни; по этим данным не разделить. REVE видит нормированный сигнал и всё равно выше (0,94), т. е. ловит больше, чем амплитуду, но влияние возраста/аппарата исключить нельзя.
5. **Контроль — здоровые люди**, задача проще клинической. **0,94 — не оценка качества для врача.** В литературе на клинических данных (контроль — пациенты без эпилепсии) ROC AUC ~0,76–0,79.

**Ограничения:** C подбиралось на тех же разбиениях, что и оценка (лёгкое завышение); возраст и пол участников неизвестны; REVE не дообучалась.

## 5. Что дальше

- Повторить ту же схему на **TUEP** (после получения доступа к TUH) — там контроль из той же клиники.
- Сравнить с **DeepEpilepsy** (готовые веса, обучена именно на задаче «эпилепсия по рутинной ЭЭГ»).
- Попробовать дообучение REVE вместо замороженных признаков.